<a href="https://colab.research.google.com/github/adelinewidyatmoko/ProjectA_Kelompok6_BanjirArticles_PBA/blob/main/notebooks/03_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation

In [ ]:
!pip install newspaper3k lxml_html_clean beautifulsoup4 -q
!pip install Sastrawi
!pip install wordcloud -q

In [ ]:
import pandas as pd
import numpy as np
import time
import random
import re
import os
import requests
from bs4 import BeautifulSoup
from newspaper import Article
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
sns.set_theme(style="whitegrid")

# 2. Text Cleanning

### 2.1 Remove URL, Emoji, punctuation, angka

In [ ]:
import string
import re
from pandas.core.arrays.categorical import contains
#Dataframe Banjir
df = pd.read_csv('data_banjir_clean.csv')

if 'Unamed:4' in df.columns:
  df = df.drop(columns=['Unamed:4'])
else:
  df

is_banjir = df['Sentimen'].str.lower() == 'banjir'
is_bandang = df['Sentimen'].str.lower() == 'bandang'

if is_banjir.any():
  df_sentimen_banjir = df.loc[is_banjir, ['Judul', 'Link', 'Sumber', 'Sentimen', 'Konten', 'Tanggal']]
  df_sentimen_banjir = df_sentimen_banjir.dropna(subset=['Konten'])
else :
  print(f"none")

if is_bandang.any():
  df_sentimen_bandang = df.loc[is_bandang, ['Judul', 'Link', 'Sumber', 'Sentimen', 'Konten', 'Tanggal']]
  df_sentimen_bandang = df_sentimen_bandang.dropna(subset=['Konten'])
else :
  print(f"none")




In [ ]:
def data_preprocessing(text):
  try:
    text = text.lower()
    clean_pattern = r'http\S+|@\S+|#\S+|[^\w\s]|\b\d+\b|\b[a-zA-Z]\b'
    text = re.sub(r'[\n\r\t]', ' ', text)
    text = re.sub(clean_pattern, ' ', text)
    text = re.sub(r'(.)\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if any(texts in string.punctuation and re.search(r'\d+|[^\w\s]|-', texts) for texts in text):
      return True
    else:
      return f"{text}"

  except Exception as e:
    return {e}

df_sentimen_banjir['Konten_clean'] =  df_sentimen_banjir['Konten'].apply(data_preprocessing)
df_sentimen_bandang['Konten_clean'] = df_sentimen_bandang['Konten'].apply(data_preprocessing)

df_prep = df_sentimen_banjir['Konten_clean'].to_string()
df_prep1 = df_sentimen_bandang['Konten_clean'].to_string()
#Clean Data
banjir_konten = data_preprocessing(df_prep)
bandang_konten = data_preprocessing(df_prep1)


In [ ]:
#Before and after stopwords
#!pip install nltk
!pip install sastrawi
import nltk
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### 2.2 Distribusi Jumlah Artikel per Tahun

In [ ]:
# Pastikan kolom Tanggal dikonversi ke tipe datetime
df['Tanggal'] = pd.to_datetime(df['Tanggal'], errors='coerce')
# Ekstrak tahun dari kolom Tanggal
df['Tahun'] = df['Tanggal'].dt.year

plt.figure(figsize=(10, 6))
# Membuat countplot untuk menghitung jumlah artikel per tahun
ax = sns.countplot(data=df, x='Tahun', palette='viridis')

plt.title('Distribusi Jumlah Artikel per Tahun', fontsize=15, fontweight='bold')
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Jumlah Artikel', fontsize=12)
plt.xticks(rotation=45)

# Menambahkan angka di atas bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

### 2.3 Distribusi Artikel Berdasarkan Kategori (Banjir vs Bandang)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Sentimen', palette='Set2')

plt.title('Distribusi Artikel Berdasarkan Kategori (Banjir vs Bandang)', fontsize=15, fontweight='bold')
plt.xlabel('Kategori / Sentimen', fontsize=12)
plt.ylabel('Jumlah Artikel', fontsize=12)

# Menambahkan angka di atas bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

### 2.4 Top 10 Sumber Berita dengan Artikel Terbanyak

In [ ]:
plt.figure(figsize=(10, 6))
# Mengambil 10 sumber terbanyak
top_10_sumber = df['Sumber'].value_counts().head(10)

sns.barplot(x=top_10_sumber.values, y=top_10_sumber.index, palette='magma')

plt.title('Top 10 Sumber Berita dengan Artikel Terbanyak', fontsize=15, fontweight='bold')
plt.xlabel('Jumlah Artikel', fontsize=12)
plt.ylabel('Sumber Berita', fontsize=12)

plt.tight_layout()
plt.show()

### 2.5 Top 100 Kata Berdasarkan Sentimen

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Menggabungkan seluruh teks Konten berdasarkan Sentimen
konten_banjir = " ".join(df[df['Sentimen'].str.lower() == 'banjir']['Konten'].dropna().astype(str))
konten_bandang = " ".join(df[df['Sentimen'].str.lower() == 'bandang']['Konten'].dropna().astype(str))

# Fungsi untuk mengambil top N kata (tokenisasi sederhana karena belum di preprocess)
def get_top_words(text, n=100):
    words = re.findall(r'\b\w+\b', text.lower())
    return Counter(words).most_common(n)

top_100_banjir = get_top_words(konten_banjir, 100)
top_100_bandang = get_top_words(konten_bandang, 100)

# Memisahkan kata dan frekuensi untuk plotting
kata_banjir, freq_banjir = zip(*top_100_banjir)
kata_bandang, freq_bandang = zip(*top_100_bandang)

# Membuat figure yang tinggi agar 100 kata bisa terbaca (tidak menumpuk)
fig, axes = plt.subplots(1, 2, figsize=(20, 25))

# Plot Top 100 Kata Banjir Biasa
sns.barplot(x=list(freq_banjir), y=list(kata_banjir), ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 100 Kata (Sentimen: Banjir)\nSebelum Preprocess', fontsize=18, fontweight='bold')
axes[0].set_xlabel('Frekuensi', fontsize=14)
axes[0].set_ylabel('Kata', fontsize=14)

# Plot Top 100 Kata Banjir Bandang
sns.barplot(x=list(freq_bandang), y=list(kata_bandang), ax=axes[1], palette='Oranges_r')
axes[1].set_title('Top 100 Kata (Sentimen: Bandang)\nSebelum Preprocess', fontsize=18, fontweight='bold')
axes[1].set_xlabel('Frekuensi', fontsize=14)
axes[1].set_ylabel('Kata', fontsize=14)

plt.tight_layout()
plt.show()

### 2.6 Wordcloud Top 100 Kata Berdasarkan Sentimen

In [ ]:
from wordcloud import WordCloud

# Generate Wordcloud menggunakan teks yang sudah digabung di cell sebelumnya
# Max_words diatur ke 100 sesuai permintaan
wc_banjir = WordCloud(width=800, height=400, max_words=100, background_color='white', colormap='Blues').generate(konten_banjir)
wc_bandang = WordCloud(width=800, height=400, max_words=100, background_color='white', colormap='Oranges').generate(konten_bandang)

# Plotting Wordcloud secara berdampingan
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Tampilan Wordcloud untuk Banjir Biasa
axes[0].imshow(wc_banjir, interpolation='bilinear')
axes[0].set_title('Wordcloud Top 100 Kata (Sentimen: Banjir)\nSebelum Preprocess', fontsize=16, fontweight='bold')
axes[0].axis('off')

# Tampilan Wordcloud untuk Banjir Bandang
axes[1].imshow(wc_bandang, interpolation='bilinear')
axes[1].set_title('Wordcloud Top 100 Kata (Sentimen: Bandang)\nSebelum Preprocess', fontsize=16, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# 3. Stopwords Removal
(Simpan versi berfore dan after untu di analisis)

In [ ]:
import string
factory = StopWordRemoverFactory()
stop_words= factory.get_stop_words()

def before_after_stopwords(text):
  stopword = factory.create_stop_word_remover()
  remove_stopwords = stopword.remove(text)
  return remove_stopwords


df_sentimen_banjir['After_stopwords'] = df_sentimen_banjir['Konten_clean'].apply(before_after_stopwords)
df_sentimen_bandang['After_stopwords'] = df_sentimen_bandang['Konten_clean'].apply(before_after_stopwords)

df_sentimen_banjir.to_csv('banjir_biasa_sentiment.csv', index=False)
df_sentimen_bandang.to_csv('banjir_bandang_sentiment.csv', index=False)

print("✅ File 'banjir_biasa_sentiment.csv' dan 'banjir_bandang_sentiment.csv' berhasil disimpan!")

✅ File 'banjir_biasa_sentiment.csv' dan 'banjir_bandang_sentiment.csv' berhasil disimpan!


In [ ]:
df_sentimen_banjir.head()
df_sentimen_bandang.head()
#konten clean (before stopwords)

,Judul,Link,Sumber,Sentimen,Konten,Tanggal,Konten_clean,After_stopwords
1279,"Update Korban Bencana Alam NTT: 128 Meninggal,...",https://news.detik.com/berita/d-5521716/update...,Detik,Bandang,Korban meninggal akibat bencana di Nusa Tengga...,2021-04-06 10:42:09,korban meningal akibat bencana di nusa tengara...,korban meningal akibat bencana nusa tengara ti...
1280,69 korban banjir bandang di Adonara ditemukan ...,https://kalteng.antaranews.com/berita/467818/6...,Antara News,Bandang,69 korban banjir bandang di Adonara ditemukan ...,NaN,korban banjir bandang di adonara ditemukan men...,korban banjir bandang adonara ditemukan mening...
1281,Korban meninggal akibat bencana di NTT jadi 18...,https://www.aa.com.tr/id/regional/korban-menin...,Lainnya,Bandang,Wakil Gubernur NTT Josef Nae menyatakan sebany...,NaN,wakil gubernur nt josef nae menyatakan sebanya...,wakil gubernur nt josef nae menyatakan sebanya...
1282,Korban Meninggal Akibat Siklon Tropis Seroja d...,https://databoks.katadata.co.id/lingkungan/sta...,Lainnya,Bandang,Bencana alam badai siklon tropis Seroja menerp...,NaN,bencana alam badai siklon tropis seroja menerp...,bencana alam badai siklon tropis seroja menerp...
1286,11 Orang Meninggal Dunia Akibat Banjir Bandang...,https://www.viva.co.id/berita/nasional/1361631...,Viva,Bandang,VIVA – Badan Penanggulangan Bencana Daerah (BP...,2021-04-05 09:00:35+07:00,viva badan penangulangan bencana daerah bpbd k...,viva badan penangulangan bencana daerah bpbd k...


# 4. Tokenization

In [ ]:
def tokenization(text):
  token = word_tokenize(text)
  return token

after_stopwords_banjir = before_after_stopwords(banjir_konten)
after_stopwords_bandang = before_after_stopwords(bandang_konten)
token_banjir = tokenization(after_stopwords_banjir)
token_bandang = tokenization(after_stopwords_bandang)
print(token_banjir)


# 5. Lemmatization

(Dilakukan setalah stemming untuk refinement)

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize

# Initialize the default stemmer
factory = StemmerFactory()
base_stemmer = factory.create_stemmer()

# --- Custom Stemming Mappings ---
# Define custom mappings for words that Sastrawi might not stem correctly.
custom_stem_map = {
    'bersihkan': 'bersih',
    # Add more custom mappings here if needed, e.g.:
    # 'kata_asli': 'hasil_stem_yang_diinginkan',
}

# Wrap the original stemmer to apply custom_stem_map
class CustomStemmer:
    def __init__(self, base_stemmer, custom_map):
        self.base_stemmer = base_stemmer
        self.custom_map = custom_map

    def stem(self, word):
        word_lower = word.lower()
        if word_lower in self.custom_map:
            return self.custom_map[word_lower]
        return self.base_stemmer.stem(word)

stemmer = CustomStemmer(base_stemmer, custom_stem_map)
# --- End Custom Stemming Mappings ---

In [ ]:
def lemmatization(text):
    if isinstance(text, str):
        words = word_tokenize(text) # Tokenize the text into individual words
        stemmed_words = [stemmer.stem(word) for word in words]
        return ' '.join(stemmed_words)
    return text # Return non-string values as is

# Apply lemmatization to the clean content columns
# new_df["After_lemma"] = new_df["After_stopwords"].apply(lemmatization)
df_sentimen_banjir['After_lemma'] = df_sentimen_banjir['After_stopwords'].apply(lemmatization)
df_sentimen_bandang['After_lemma'] = df_sentimen_bandang['After_stopwords'].apply(lemmatization)


In [ ]:
print(f"Test word 'bersihkan' after lemmatization: {lemmatization('bersihkan')}")
print("\nHead of df_sentimen_banjir after lemmatization:")
print(df_sentimen_banjir[['After_stopwords', 'After_lemma']].head())

# 1. Menggabungkan kedua dataframe menjadi satu
df_master_lemmatized = pd.concat([df_sentimen_banjir, df_sentimen_bandang], ignore_index=True)

# 2. Menyimpan dataframe gabungan ke dalam 1 file CSV
file_name = 'data_banjir_master_lemmatized.csv'
df_master_lemmatized.to_csv(file_name, index=False)

print(f"✅ Data berhasil digabung (Total baris: {len(df_master_lemmatized)})")
print(f"✅ File berhasil disimpan dengan nama: {file_name}")